<h1> SHAP-Analysis – AutoGluon</h1>

Before starting the script, you need to initialise the persistant_predict_server_autogluon.py file.
Check your Root folder & your specific AutoGluon DockerID

docker cp ROOTFOLDER\persistent_predict_server_autogluon.py DOCKERID:/workspace/persistent_predict_server_autogluon.py
docker start DOCKERID

In [4]:
import os
import json
import subprocess
import uuid

import numpy as np
import pandas as pd
import shap

In [5]:
FRAMEWORK = "autogluon"

HOST_BASE = r"E:\StudiumMasterarbeit"
SAVED_MODELS_DIR = os.path.join(HOST_BASE, "saved_models", "autogluon")

CONTAINER_ID = "3caab0b22cc7"
PERSISTENT_SERVER_IN_CONTAINER = "/workspace/persistent_predict_server_autogluon.py"
CONTAINER_MOUNT_PREFIX = "/workspace"
TMP_DIRNAME = "tmp_shap_bridge_autogluon"

N_BACKGROUND = 50
N_EXPLAIN = 30
SKIP_IF_RESULTS_EXIST = True

OUTPUT_DIR = os.path.join(HOST_BASE, "shap_results", FRAMEWORK)
TMP_DIR = os.path.join(HOST_BASE, TMP_DIRNAME)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

tasks = sorted(
    d for d in os.listdir(SAVED_MODELS_DIR)
    if os.path.isdir(os.path.join(SAVED_MODELS_DIR, d))
)

def container_path(host_path):
    return CONTAINER_MOUNT_PREFIX + "/" + os.path.relpath(
        host_path, HOST_BASE
    ).replace("\\", "/")

print(f"{len(tasks)} Aufgaben gefunden.")


15 Aufgaben gefunden.


In [ ]:
# SHAP-Werte berechnen als Loop, der durch alle Aufgaben iteriert. Ergebnisse werden in OUTPUT_DIR gespeichert.
for task_name in tasks:
    task_dir = os.path.join(SAVED_MODELS_DIR, task_name)
    out_dir = os.path.join(OUTPUT_DIR, task_name)

    # Bereits berechnete Aufgaben überspringen
    if SKIP_IF_RESULTS_EXIST and os.path.exists(
        os.path.join(out_dir, "shap_values.npy")
    ):
        print(f"[{task_name}] übersprungen – Ergebnis existiert bereits.")
        continue

    os.makedirs(out_dir, exist_ok=True)

    print(f"\n[{task_name}]")

    with open(os.path.join(task_dir, "feature_names.json")) as f:
        feature_names = json.load(f)

    with open(os.path.join(task_dir, "meta.json")) as f:
        meta = json.load(f)

    train_df = pd.read_csv(os.path.join(task_dir, "train_data.csv"))
    test_df = pd.read_csv(os.path.join(task_dir, "test_data.csv"))
    X_train = train_df[feature_names].to_numpy()
    X_test = test_df[feature_names].to_numpy()

    predictor_path = meta["predictor_path"]
    feature_names_container = container_path(
        os.path.join(task_dir, "feature_names.json")
    )

    background = shap.sample(
        X_train, min(N_BACKGROUND, len(X_train)), random_state=42
    )
    X_explain = X_test[:min(N_EXPLAIN, len(X_test))]
    max_evals = max(500, 2 * len(feature_names) + 1)

    proc = subprocess.Popen(
        [
            "docker", "exec", "-i", CONTAINER_ID,
            "python3", "-u",
            PERSISTENT_SERVER_IN_CONTAINER,
            predictor_path, feature_names_container
        ],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        text=True,
        bufsize=1
    )

    try:
        if proc.stdout.readline().strip() != "READY":
            raise RuntimeError("Predict-Server nicht bereit.")

        def predict(X):
            uid = uuid.uuid4().hex
            host_in = os.path.join(TMP_DIR, f"in_{uid}.npy")
            host_out = os.path.join(TMP_DIR, f"out_{uid}.npy")

            try:
                np.save(host_in, np.asarray(X))

                proc.stdin.write(
                    f"{container_path(host_in)}\t{container_path(host_out)}\n"
                )
                proc.stdin.flush()

                if proc.stdout.readline().strip() != "OK":
                    raise RuntimeError("Predict-Server-Fehler.")

                return np.load(host_out)
            finally:
                if os.path.exists(host_in):
                    os.remove(host_in)
                if os.path.exists(host_out):
                    os.remove(host_out)

        print(
            f"  {len(X_train)} Trainingsdaten | "
            f"{len(X_explain)} zu erklärende Samples | "
            f"{len(feature_names)} Merkmale"
        )

        explainer = shap.Explainer(
            predict,
            background,
            feature_names=feature_names,
            seed=42
        )
        shap_values = explainer(X_explain, max_evals=max_evals)

        # Efficiency-Check: SHAP-Werte + Base Value sollten der
        # Modellvorhersage entsprechen.
        y_pred_explained = predict(X_explain)
        base_values = np.asarray(shap_values.base_values)
        efficiency_gap = y_pred_explained - (
            shap_values.values.sum(axis=1) + base_values
        )

        print(
            f"  Efficiency-Check: mean|gap| = "
            f"{np.mean(np.abs(efficiency_gap)):.6f}"
        )

    finally:
        try:
            proc.stdin.write("EXIT\n")
            proc.stdin.flush()
        except Exception:
            pass

        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.terminate()
            proc.wait(timeout=10)

    np.save(os.path.join(out_dir, "shap_values.npy"), shap_values.values)
    np.save(os.path.join(out_dir, "X_explained.npy"), X_explain)
    np.save(os.path.join(out_dir, "base_values.npy"), base_values)
    np.save(os.path.join(out_dir, "y_pred_explained.npy"), y_pred_explained)

    with open(os.path.join(out_dir, "feature_names.json"), "w") as f:
        json.dump(list(feature_names), f)

    print(f"  Gespeichert: {out_dir}")